<a href="https://colab.research.google.com/github/Nilufar-Komil/Applications-of-the-Generalized-Stiefel-Manifold-in-NN/blob/READY/WRN_28_10_gs_all_with_B_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-

import os
import sys
sys.path.append(os.getcwd())
from typing import Optional
import torch.nn as nn
import random
import torch # Main PyTorch Library
import torchvision # Pytorch library for image analysis
from torchvision import transforms  # Transform function used to modify and preprocess all the images
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
from torch.utils.data import random_split #Used to split data sets
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

Device available:  cuda


In [ ]:
import torchvision.transforms as transforms

# Normalize all images: shift pixel values within -1 and 1
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

full_train_dataset = torchvision.datasets.CIFAR10(
    root="./data",                  # save/load dataset in this folder
    train=True,                     # use training data
    download=True,                  # download if needed
    transform=transform             # apply transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",                  # save/load dataset in this folder
    train=False,                    # use test data
    download=True,                  # download if needed
    transform=transform             # apply transform
)

# print size of each data sets
print("Train size:", len(full_train_dataset))
print("Test size:", len(test_dataset))


100%|██████████| 170M/170M [00:03<00:00, 49.4MB/s]


Train size: 50000
Test size: 10000


In [ ]:
# split full training data set to training and validation parts
train_size = int(0.8 * len(full_train_dataset))   # 80% for training
val_size = len(full_train_dataset) - train_size   # 20% for validation

train_dataset, val_dataset = random_split(
    full_train_dataset,             # dataset to split
    [train_size, val_size],         # sizes of split parts
    generator=torch.Generator().manual_seed(42)   # makes the train/validation split stay the same every time
)

# printing each data sets sizes
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))


Train size: 40000
Val size: 10000
Test size: 10000


In [ ]:
batch_size = 128


torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = False


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ProductGStiefelConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size,
        diffeo,
        stride: int = 1,
        padding: int = 0,
        bias: bool = False,
    ):
        super().__init__()

        if isinstance(kernel_size, int):
            self.kernel_size = (kernel_size, kernel_size)
        else:
            self.kernel_size = kernel_size

        kh, kw = self.kernel_size

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride
        self.padding = padding
        self.diffeo = diffeo

        self.n = in_channels * kh * kw
        self.p = out_channels
        self.rank = diffeo.rank

        if self.p > self.n:
            raise ValueError(
                f"Infeasible layer: out_channels={self.p} > n={self.n}"
            )
        if not (self.p <= self.rank <= self.n):
            raise ValueError(
                f"Expected p <= rank <= n, got p={self.p}, rank={self.rank}, n={self.n}"
            )

        X_std, Z_std = diffeo.random_product_point(self.p)

        self.X = nn.Parameter(X_std.t().contiguous())
        self.Z = nn.Parameter(Z_std.t().contiguous())

        if bias:
            self.bias = nn.Parameter(
                torch.zeros(out_channels, device=self.X.device, dtype=self.X.dtype)
            )
        else:
            self.register_parameter("bias", None)

    def reconstruct_weight(self) -> torch.Tensor:
        X_std = self.X.t()
        Z_std = self.Z.t()

        W_flat = self.diffeo.phi_inverse(X_std, Z_std)
        if not isinstance(W_flat, torch.Tensor):
            raise TypeError("phi_inverse must return a torch.Tensor")

        kh, kw = self.kernel_size
        W = W_flat.t().contiguous().view(
            self.out_channels,
            self.in_channels,
            kh,
            kw,
        )
        return W

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        W = self.reconstruct_weight()
        return F.conv2d(
            x,
            W,
            bias=self.bias,
            stride=self.stride,
            padding=self.padding,
        )

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class WRNBasicBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        dropout=0.3,
        conv0_layer=None,
        conv1_layer=None,
    ):
        super().__init__()
        self.equal_in_out = (in_channels == out_channels)
        self.dropout = dropout

        self.bn0 = nn.BatchNorm2d(in_channels)

        self.conv0 = conv0_layer if conv0_layer is not None else nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv1 = conv1_layer if conv1_layer is not None else nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1, bias=False
        )

        self.convdim = None
        if not self.equal_in_out:
            self.convdim = nn.Conv2d(
                in_channels, out_channels,
                kernel_size=1, stride=stride, padding=0, bias=False
            )

        self.reset_parameters()

    def reset_parameters(self):
        if isinstance(self.conv0, nn.Conv2d):
            nn.init.orthogonal_(self.conv0.weight)
        if isinstance(self.conv1, nn.Conv2d):
            nn.init.orthogonal_(self.conv1.weight)
        if self.convdim is not None:
            nn.init.orthogonal_(self.convdim.weight)

        nn.init.ones_(self.bn0.weight)
        nn.init.zeros_(self.bn0.bias)
        nn.init.ones_(self.bn1.weight)
        nn.init.zeros_(self.bn1.bias)

    def forward(self, x):
        o1 = F.relu(self.bn0(x), inplace=True)

        y = self.conv0(o1)
        o2 = F.relu(self.bn1(y), inplace=True)
        o2 = F.dropout(o2, p=self.dropout, training=self.training)
        z = self.conv1(o2)

        shortcut = self.convdim(o1) if self.convdim is not None else x
        return z + shortcut

In [ ]:
class WRNGroup(nn.Module):
    def __init__(
        self,
        num_blocks,
        in_channels,
        out_channels,
        first_stride,
        dropout=0.3,
        block_builder=None,
        group_index=None,
    ):
        super().__init__()
        blocks = []

        for i in range(num_blocks):
            stride = first_stride if i == 0 else 1
            block_in = in_channels if i == 0 else out_channels

            if block_builder is None:
                block = WRNBasicBlock(
                    in_channels=block_in,
                    out_channels=out_channels,
                    stride=stride,
                    dropout=dropout,
                )
            else:
                block = block_builder(
                    group_index=group_index,
                    block_index=i,
                    in_channels=block_in,
                    out_channels=out_channels,
                    stride=stride,
                    dropout=dropout,
                )

            blocks.append(block)

        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)

In [ ]:
class WideResNet(nn.Module):
    def __init__(
        self,
        depth=28,
        width=10,
        num_classes=10,
        dropout=0.3,
        stem_layer=None,
        block_builder=None,
    ):
        super().__init__()
        assert (depth - 4) % 6 == 0, "depth should be 6n+4"

        n = (depth - 4) // 6
        widths = [16 * width, 32 * width, 64 * width]

        self.conv0 = stem_layer if stem_layer is not None else nn.Conv2d(
            3, 16, kernel_size=3, stride=1, padding=1, bias=False
        )

        self.group0 = WRNGroup(
            num_blocks=n,
            in_channels=16,
            out_channels=widths[0],
            first_stride=1,
            dropout=dropout,
            block_builder=block_builder,
            group_index=0,
        )
        self.group1 = WRNGroup(
            num_blocks=n,
            in_channels=widths[0],
            out_channels=widths[1],
            first_stride=2,
            dropout=dropout,
            block_builder=block_builder,
            group_index=1,
        )
        self.group2 = WRNGroup(
            num_blocks=n,
            in_channels=widths[1],
            out_channels=widths[2],
            first_stride=2,
            dropout=dropout,
            block_builder=block_builder,
            group_index=2,
        )

        self.bn = nn.BatchNorm2d(widths[2])
        self.fc = nn.Linear(widths[2], num_classes)

        self.reset_parameters()

    def reset_parameters(self):
        if isinstance(self.conv0, nn.Conv2d):
            nn.init.orthogonal_(self.conv0.weight)

        nn.init.ones_(self.bn.weight)
        nn.init.zeros_(self.bn.bias)

        nn.init.normal_(self.fc.weight, mean=0.0, std=2 / (self.fc.in_features ** 0.5))
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        x = self.conv0(x)
        x = self.group0(x)
        x = self.group1(x)
        x = self.group2(x)
        x = F.relu(self.bn(x), inplace=True)
        x = F.avg_pool2d(x, 8, 1, 0)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
#a helper that decides whether a layer should be constrained
def is_feasible_gstiefel_conv(in_channels, out_channels, kernel_size, rank):
    n = in_channels * kernel_size * kernel_size
    p = out_channels
    return (p <= n) and (p <= rank <= n)


In [ ]:
def make_block_builder(diffeo_factory):
    """
    diffeo_factory(in_channels, out_channels, kernel_size, group_index, block_index, conv_name)
    should return a diffeo object with appropriate B/rank for that layer.
    """

    def build_block(group_index, block_index, in_channels, out_channels, stride, dropout):
        # conv0: first conv inside block
        # skip constraint for the known infeasible WRN layer: group0.block0.conv0 (16 -> 160, 3x3)
        use_gs_conv0 = not (group_index == 0 and block_index == 0)

        conv0_layer = None
        if use_gs_conv0:
            diffeo0 = diffeo_factory(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                group_index=group_index,
                block_index=block_index,
                conv_name="conv0",
            )
            if is_feasible_gstiefel_conv(in_channels, out_channels, 3, diffeo0.rank):
                conv0_layer = ProductGStiefelConv2d(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=3,
                    diffeo=diffeo0,
                    stride=stride,
                    padding=1,
                    bias=False,
                )

        # conv1: always out_channels -> out_channels, usually feasible in WRN-28-10
        conv1_layer = None
        diffeo1 = diffeo_factory(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=3,
            group_index=group_index,
            block_index=block_index,
            conv_name="conv1",
        )
        if is_feasible_gstiefel_conv(out_channels, out_channels, 3, diffeo1.rank):
            conv1_layer = ProductGStiefelConv2d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                diffeo=diffeo1,
                stride=1,
                padding=1,
                bias=False,
            )

        return WRNBasicBlock(
            in_channels=in_channels,
            out_channels=out_channels,
            stride=stride,
            dropout=dropout,
            conv0_layer=conv0_layer,
            conv1_layer=conv1_layer,
        )

    return build_block

In [ ]:
def make_stem_layer(diffeo_factory):
    diffeo = diffeo_factory(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        group_index=-1,
        block_index=-1,
        conv_name="stem",
    )
    return ProductGStiefelConv2d(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        diffeo=diffeo,
        stride=1,
        padding=1,
        bias=False,
    )

In [ ]:

# import future annotations so type hints can reference the class itself safely
import torch


# ------------------------------------------------------------
# Helper function:
# return the symmetric part of a square matrix
# ------------------------------------------------------------
def sym(A: torch.Tensor) -> torch.Tensor:
    # compute (A + A^T) / 2
    return 0.5 * (A + A.T)


# ------------------------------------------------------------
# Diffeomorphism between generalized Stiefel St_B and a product
# manifold consisting of:
#   - a standard Stiefel part
#   - a Euclidean part
#
# If B = Q diag(Lambda, 0) Q^T with rank r, and W is written as
#
#   Q^T W = [W1]
#           [W2]
#
# then define
#
#   X = Lambda^(1/2) W1
#   Z = W2
#
# and reconstruct by
#
#   W1 = Lambda^(-1/2) X
#   W  = Q1 W1 + Q2 Z
#
# For W in St_B, the Stiefel part satisfies X^T X = I.
# ------------------------------------------------------------
class GeneralizedStiefelDiffeomorphismTorch:
    def __init__(self, B: torch.Tensor, tol: float = 1e-10):
        # store the matrix B
        self.B = B

        # numerical tolerance used for rank decisions and checks
        self.tol = tol

        # store device and dtype for creating compatible tensors later
        self.device = B.device
        self.dtype = B.dtype

        # check that B is square
        if B.ndim != 2 or B.shape[0] != B.shape[1]:
            raise ValueError("B must be a square matrix.")

        # store ambient dimension n
        self.n = B.shape[0]

        # enforce symmetry numerically to avoid tiny asymmetry problems
        Bsym = sym(B)

        # eigen-decomposition of symmetric B
        # eigh is used because B is symmetric
        evals, Q = torch.linalg.eigh(Bsym)

        # sort eigenvalues from largest to smallest
        idx = torch.argsort(evals, descending=True)
        evals = evals[idx]
        Q = Q[:, idx]

        # B must be positive semidefinite, so no significantly negative eigenvalues
        if torch.any(evals < -tol):
            raise ValueError("B must be positive semidefinite.")

        # numerical rank = number of eigenvalues greater than tolerance
        rank = int((evals > tol).sum().item())

        # store rank(B)
        self.rank = rank

        # store full eigenvector matrix
        self.Q = Q

        # store sorted eigenvalues
        self.evals = evals

        # split Q into image-space basis Q1 and null-space basis Q2
        self.Q1 = Q[:, :rank]
        self.Q2 = Q[:, rank:]

        # positive eigenvalues only
        if rank > 0:
            # keep the strictly positive eigenvalues
            lam = evals[:rank]

            # diagonal matrix Lambda
            self.Lambda = torch.diag(lam)

            # diagonal square root Lambda^(1/2)
            self.Lambda_half = torch.diag(torch.sqrt(lam))

            # diagonal inverse square root Lambda^(-1/2)
            self.Lambda_half_inv = torch.diag(1.0 / torch.sqrt(lam))
        else:
            # if rank is zero, create empty matrices with correct shapes
            self.Lambda = torch.empty((0, 0), device=self.device, dtype=self.dtype)
            self.Lambda_half = torch.empty((0, 0), device=self.device, dtype=self.dtype)
            self.Lambda_half_inv = torch.empty((0, 0), device=self.device, dtype=self.dtype)

    # --------------------------------------------------------
    # Check whether W belongs to the generalized Stiefel manifold:
    #     W^T B W = I
    # Here W must have shape (n, p)
    # --------------------------------------------------------
    def in_generalized_stiefel(self, W: torch.Tensor) -> bool:
        # check ambient dimension
        if W.ndim != 2 or W.shape[0] != self.n:
            return False

        # number of columns p
        p = W.shape[1]

        # compute W^T B W
        gram = W.T @ self.B @ W

        # compare against identity
        I = torch.eye(p, device=W.device, dtype=W.dtype)

        # return whether the constraint is satisfied up to tolerance
        return torch.norm(gram - I) <= 1e2 * self.tol

    # --------------------------------------------------------
    # Map W in St_B to product coordinates (X, Z)
    #
    # Input:
    #   W : shape (n, p)
    #
    # Output:
    #   X : shape (rank, p)  with X^T X = I if W in St_B
    #   Z : shape (n-rank, p)
    # --------------------------------------------------------
    def phi(self, W: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # check shape
        if W.ndim != 2 or W.shape[0] != self.n:
            raise ValueError(f"W must have shape ({self.n}, p).")

        # coordinates of W in the Q basis
        QtW = self.Q.T @ W

        # top block in the range of B
        W1 = QtW[:self.rank, :]

        # bottom block in the null-space of B
        W2 = QtW[self.rank:, :]

        # Stiefel part:
        # X = Lambda^(1/2) W1
        if self.rank > 0:
            X = self.Lambda_half @ W1
        else:
            # if rank is zero, X is empty with 0 rows
            X = torch.empty((0, W.shape[1]), device=W.device, dtype=W.dtype)

        # Euclidean part:
        # Z = W2
        Z = W2

        # return product coordinates
        return X, Z

    # --------------------------------------------------------
    # Inverse map from product coordinates (X, Z) back to W
    #
    # Input:
    #   X : shape (rank, p)
    #   Z : shape (n-rank, p)
    #
    # Output:
    #   W : shape (n, p)
    # --------------------------------------------------------
    def phi_inverse(self, X: torch.Tensor, Z: torch.Tensor) -> torch.Tensor:
        # check dimensions of X
        if X.ndim != 2:
            raise ValueError("X must be a matrix of shape (rank, p).")

        # check dimensions of Z
        if Z.ndim != 2:
            raise ValueError("Z must be a matrix of shape (n-rank, p).")

        # both product blocks must have the same number of columns p
        if X.shape[1] != Z.shape[1]:
            raise ValueError("X and Z must have the same number of columns.")

        # X must have rank rows
        if X.shape[0] != self.rank:
            raise ValueError(f"X must have shape ({self.rank}, p).")

        # Z must have n-rank rows
        if Z.shape[0] != self.n - self.rank:
            raise ValueError(f"Z must have shape ({self.n - self.rank}, p).")

        # reconstruct W1 = Lambda^(-1/2) X
        if self.rank > 0:
            W1 = self.Lambda_half_inv @ X
        else:
            # if rank is zero, W1 is empty
            W1 = torch.empty((0, X.shape[1]), device=X.device, dtype=X.dtype)

        # reconstruct W = Q1 W1 + Q2 Z
        W = self.Q1 @ W1 + self.Q2 @ Z

        # return ambient matrix W in the original coordinates
        return W

    # --------------------------------------------------------
    # Create a valid random point in the product manifold
    #
    # Input:
    #   p : number of columns
    #
    # Output:
    #   X : shape (rank, p), satisfying X^T X = I
    #   Z : shape (n-rank, p), arbitrary Euclidean block
    #
    # Important geometric condition:
    #   for X^T X = I to be possible, we need p <= rank
    # --------------------------------------------------------
    def random_product_point(self, p: int) -> tuple[torch.Tensor, torch.Tensor]:
        # Stiefel part needs rank >= p
        if p > self.rank:
            raise ValueError(
                f"Cannot create a Stiefel point with p={p} columns when rank(B)={self.rank}. "
                "You need p <= rank(B)."
            )

        # ----------------------------------------------------
        # build X in St(rank, p), meaning:
        #   X has shape (rank, p)
        #   X^T X = I_p
        #
        # easiest way:
        #   sample a random rank x p matrix
        #   take reduced QR
        # ----------------------------------------------------
        A = torch.randn(self.rank, p, device=self.device, dtype=self.dtype)

        # reduced QR gives Qx with orthonormal columns
        Qx, Rx = torch.linalg.qr(A, mode="reduced")

        # fix column signs so QR is deterministic up to sign
        d = torch.diagonal(Rx, 0)
        s = torch.sign(d)
        s[s == 0] = 1
        Qx = Qx * s.unsqueeze(0)

        # this is the standard Stiefel variable
        X = Qx

        # Euclidean block can start from zeros
        Z = torch.zeros(self.n - self.rank, p, device=self.device, dtype=self.dtype)

        # return a valid product point
        return X, Z

    # --------------------------------------------------------
    # Initialize product coordinates from an existing W in St_B
    #
    # This is useful if you already have a valid generalized
    # Stiefel point and want to convert it into (X, Z).
    # --------------------------------------------------------
    def init_from_W(self, W: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # optionally verify membership
        if not self.in_generalized_stiefel(W):
            raise ValueError("Input W does not satisfy W^T B W = I up to tolerance.")

        # map to product coordinates
        return self.phi(W)

    # --------------------------------------------------------
    # Print a compact summary of the decomposition
    # --------------------------------------------------------
    def print_summary(self) -> None:
        # print ambient dimension
        print(f"n = {self.n}")

        # print rank of B
        print(f"rank(B) = {self.rank}")

        # print positive eigenvalues
        print("positive eigenvalues =", self.evals[:self.rank])

        # print shapes of decomposition blocks
        print("Q1 shape =", tuple(self.Q1.shape))
        print("Q2 shape =", tuple(self.Q2.shape))

In [ ]:
import math
import torch

dtype = torch.float32

def make_B_for_conv(n, p, device, dtype=torch.float32):
    target_rank = math.ceil(1.1 * p)

    if target_rank > n:
        raise ValueError(
            f"Cannot set rank(B)={target_rank} because n={n} and p={p}."
        )

    zero_count = n - target_rank

    d = torch.linspace(0.1, 2.5, n, device=device, dtype=dtype)
    d[:zero_count] = 0.0
    B = torch.diag(d)

    return B, target_rank


def diffeo_factory(in_channels, out_channels, kernel_size, group_index, block_index, conv_name):
    n = in_channels * kernel_size * kernel_size
    p = out_channels

    if p > n:
        raise ValueError(
            f"Infeasible layer: p={p} > n={n} for {conv_name}, "
            f"group={group_index}, block={block_index}"
        )

    B, target_rank = make_B_for_conv(n, p, device=device, dtype=torch.float32)

    diffeo = GeneralizedStiefelDiffeomorphismTorch(B=B)

    layer_name = (
        "conv0" if conv_name == "stem"
        else f"group{group_index}.block{block_index}.{conv_name}"
    )

    if diffeo.rank < p:
        raise ValueError(
            f"rank(B)={diffeo.rank} < p={p} for {layer_name}"
        )

    print(f"{layer_name}: n={n}, p={p}, target_rank={target_rank}, actual_rank={diffeo.rank}")

    return diffeo

In [ ]:
stem_layer = make_stem_layer(diffeo_factory)
block_builder = make_block_builder(diffeo_factory)

conv0: n=27, p=16, target_rank=18, actual_rank=18


In [ ]:
model = WideResNet(
    depth=28,
    width=10,
    num_classes=10,
    dropout=0.3,
    stem_layer=stem_layer,
    block_builder=block_builder,
)

group0.block0.conv1: n=1440, p=160, target_rank=176, actual_rank=176
group0.block1.conv0: n=1440, p=160, target_rank=176, actual_rank=176
group0.block1.conv1: n=1440, p=160, target_rank=176, actual_rank=176
group0.block2.conv0: n=1440, p=160, target_rank=176, actual_rank=176
group0.block2.conv1: n=1440, p=160, target_rank=176, actual_rank=176
group0.block3.conv0: n=1440, p=160, target_rank=176, actual_rank=176
group0.block3.conv1: n=1440, p=160, target_rank=176, actual_rank=176
group1.block0.conv0: n=1440, p=320, target_rank=352, actual_rank=352
group1.block0.conv1: n=2880, p=320, target_rank=352, actual_rank=352
group1.block1.conv0: n=2880, p=320, target_rank=352, actual_rank=352
group1.block1.conv1: n=2880, p=320, target_rank=352, actual_rank=352
group1.block2.conv0: n=2880, p=320, target_rank=352, actual_rank=352
group1.block2.conv1: n=2880, p=320, target_rank=352, actual_rank=352
group1.block3.conv0: n=2880, p=320, target_rank=352, actual_rank=352
group1.block3.conv1: n=2880, p=320

In [ ]:

# collect all constrained ProductGStiefelConv2d layers automatically
constrained_layers = []
for name, module in model.named_modules():
    if isinstance(module, ProductGStiefelConv2d):
        constrained_layers.append((name, module))


# Stiefel variables X
stiefel_params = [module.X for _, module in constrained_layers]

# Euclidean product variables Z
product_z_params = [module.Z for _, module in constrained_layers]


# exclude X and Z from the ordinary parameter list
product_param_ids = {id(p) for p in stiefel_params + product_z_params}

other_params = [
    p for p in model.parameters()
    if id(p) not in product_param_ids
]

# final Euclidean parameter list
euclid_params = product_z_params + other_params


print("number of constrained layers =", len(constrained_layers))
print("number of Stiefel parameter tensors =", len(stiefel_params))
print("number of Euclidean parameter tensors =", len(euclid_params))

number of constrained layers = 24
number of Stiefel parameter tensors = 24
number of Euclidean parameter tensors = 80


In [ ]:
#utils_modify
import math
import torch
import torch.nn as nn
import torch.cuda.comm as comm
from torch.nn.parallel._functions import Broadcast
from torch.nn.parallel import scatter, parallel_apply, gather
from functools import partial
from torch.autograd import Variable
from nested_dict import nested_dict
from collections import OrderedDict


def cast(params, dtype='float', device='cuda'):
    if isinstance(params, dict):
        return {k: cast(v, dtype=dtype, device=device) for k, v in params.items()}
    else:
        tensor = params.to(device) if device is not None else params
        return getattr(tensor, dtype)()


def conv_params(ni, no, k=1, g=1):
    assert ni % g == 0
    w = nn.init.orthogonal_(torch.empty(no, ni // g, k, k))
    return cast(w)


def linear_params(ni, no):
    return cast(dict(
        weight=torch.empty(no, ni).normal_(0, 2 / math.sqrt(ni)),
        bias=torch.zeros(no)
    ))


def bnparams(n):
    return cast(dict(
        weight=torch.ones(n),
        bias=torch.zeros(n)
    ))


def bnstats(n):
    return cast(dict(
        running_mean=torch.zeros(n),
        running_var=torch.ones(n)
    ))


def data_parallel(f, input, params, stats, mode, device_ids, output_device=None):
    if output_device is None:
        output_device = device_ids[0]

    if len(device_ids) == 1:
        return f(input, params, stats, mode)

    def replicate(param_dict, g):
        replicas = [{} for _ in device_ids]
        for k, v in param_dict.items():
            for i, u in enumerate(g(v)):
                replicas[i][k] = u
        return replicas

    params_replicas = replicate(params, lambda x: Broadcast(device_ids)(x))
    stats_replicas = replicate(stats, lambda x: comm.broadcast(x, device_ids))

    replicas = [
        partial(f, params=p, stats=s, mode=mode)
        for p, s in zip(params_replicas, stats_replicas)
    ]
    inputs = scatter([input], device_ids)
    outputs = parallel_apply(replicas, inputs)
    return gather(outputs, output_device)


def flatten_params(params):
    flat_params = OrderedDict()
    for keys, v in nested_dict(params).items_flat():
        if v is not None:
            flat_params['.'.join(keys)] = Variable(v, requires_grad=True)
    return flat_params


def flatten_stats(stats):
    flat_stats = OrderedDict()
    for keys, v in nested_dict(stats).items_flat():
        flat_stats['.'.join(keys)] = v
    return flat_stats


def matrix_norm_one(W):
    out = torch.abs(W)
    out = torch.sum(out, dim=0)
    out = torch.max(out)
    return out

In [ ]:
#gutils_modify
import torch


def norm(v, dim=1):
    assert len(v.size()) == 2
    return v.norm(p=2, dim=dim, keepdim=True)


def unit(v, dim=1, eps=1e-8):
    vnorm = norm(v, dim)
    return v / vnorm.add(eps), vnorm


def xTy(x, y):
    assert len(x.size()) == 2 and len(y.size()) == 2, 'xTy'
    return torch.sum(x * y, dim=1, keepdim=True)


def clip_by_norm(v, clip_norm):
    v_norm = norm(v)
    scale = torch.ones_like(v_norm)
    mask = v_norm > clip_norm
    scale[mask] = clip_norm / v_norm[mask]
    return v * scale


def sym_matrix(y):  # y n-by-n
    assert y.size(0) == y.size(1)
    return (y + y.t()) / 2


def skew_matrix(y):  # y n-by-n
    assert y.size(0) == y.size(1)
    return (y - y.t()) / 2


def stiefel_proj_tan(y, g):  # y,g p-by-n, p <= n
    p, n = y.size()
    skew = skew_matrix(torch.matmul(y, g.t()))
    reflect = torch.matmul(y.t(), y)
    identity = torch.eye(n, device=y.device, dtype=y.dtype)
    reflect = identity - reflect
    tan_vec = torch.matmul(y.t(), skew) + torch.matmul(reflect, g.t())
    return tan_vec.t()


def stiefel_proj_norm(y, g):  # y,g p-by-n, p <= n
    sym = sym_matrix(torch.matmul(y, g.t()))
    norm_vec = torch.matmul(y.t(), sym)
    return norm_vec.t()


def polar_retraction(tan_vec):  # tan_vec, p-by-n, p <= n
    p, n = tan_vec.size()
    U, S, Vh = torch.linalg.svd(tan_vec, full_matrices=False)
    return torch.matmul(U, Vh)


def qr_retraction(tan_vec):  # tan_vec, p-by-n, p <= n
    q, r = torch.linalg.qr(tan_vec.t(), mode='reduced')  # n-by-p
    d = torch.diagonal(r, 0)
    ph = d.sign()
    ph[ph == 0] = 1
    q = q * ph.unsqueeze(0).expand_as(q)
    return q.t()


def Cayley_loop(X, W, tan_vec, t):
    # X: n-by-p
    # W: n-by-n skew-symmetric
    # tan_vec: n-by-p
    Y = X + t * tan_vec
    for _ in range(5):
        Y = X + t * torch.matmul(W, 0.5 * (X + Y))
    return Y.t()


def check_identity(X):  # X: n-by-p
    n, p = X.size()
    res = torch.eye(p, device=X.device, dtype=X.dtype) - torch.mm(X.t(), X)
    print('n={0}, p={1}, res norm={2}'.format(n, p, torch.norm(res)))


def stiefel_transport(y, g):  # y,g p-by-n, p <= n
    return stiefel_proj_tan(y, g)


def gproj(y, g, normalize=False):
    if normalize:
        y, _ = unit(y)

    yTg = xTy(y, g)
    return g - (yTg * y)


def gexp(y, h, normalize=False):
    if normalize:
        y, _ = unit(y)
        h = gproj(y, h)

    u, hnorm = unit(h)
    return y * hnorm.cos() + u * hnorm.sin()


# parallel translation of tangent vector h1 toward h2
# both h1 and h2 are tangent vectors on y
def gpt2(y, h1, h2, normalize=False):
    if normalize:
        h1 = gproj(y, h1)
        h2 = gproj(y, h2)

    u, unorm = unit(h2)
    uTh1 = xTy(u, h1)
    return h1 - uTh1 * (unorm.sin() * y + (1 - unorm.cos()) * u)


# parallel translation if h1 = h2
def gpt(y, h, normalize=False):
    if normalize:
        h = gproj(y, h)

    u, unorm = unit(h)
    return (u * unorm.cos() - y * unorm.sin()) * unorm

In [ ]:
# stifel_optimizer_modify
import torch
from torch.optim.optimizer import Optimizer, required
import numpy as np
import random

import pdb

episilon = 1e-8


class SGDG(Optimizer):
    r"""This optimizer updates variables with two different routines
        based on the boolean variable 'stiefel'.

        If stiefel is True, the variables will be updated by SGD-G proposed
        as decorrelated weight matrix.

        If stiefel is False, the variables will be updated by SGD.
        This routine was taken from https://github.com/pytorch/pytorch/blob/master/torch/optim/sgd.py.

    Args:
        params (iterable): iterable of parameters to optimize or dicts defining
            parameter groups

        -- common parameters
        lr (float): learning rate
        momentum (float, optional): momentum factor (default: 0)
        stiefel (bool, optional): whether to use SGD-G (default: False)

        -- parameters in case stiefel is False
        weight_decay (float, optional): weight decay (L2 penalty) (default: 0)
        dampening (float, optional): dampening for momentum (default: 0)
        nesterov (bool, optional): enables Nesterov momentum (default: False)

        -- parameters in case stiefel is True
        omega (float, optional): orthogonality regularization factor (default: 0)
        grad_clip (float, optional): threshold for gradient norm clipping (default: None)
    """

    def __init__(self, params, lr=required, momentum=0, dampening=0,
                 weight_decay=0, nesterov=False,
                 stiefel=False, omega=0, grad_clip=None):
        defaults = dict(
            lr=lr,
            momentum=momentum,
            dampening=dampening,
            weight_decay=weight_decay,
            nesterov=nesterov,
            stiefel=stiefel,
            omega=omega,
            grad_clip=grad_clip
        )
        if nesterov and (momentum <= 0 or dampening != 0):
            raise ValueError("Nesterov momentum requires a momentum and zero dampening")
        super(SGDG, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(SGDG, self).__setstate__(state)
        for group in self.param_groups:
            group.setdefault('nesterov', False)

    @torch.no_grad()
    def step(self, closure=None):
        """Performs a single optimization step."""
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            momentum = group['momentum']
            stiefel = group['stiefel']
            weight_decay = group['weight_decay']
            dampening = group['dampening']
            nesterov = group['nesterov']

            for p in group['params']:
                if p.grad is None:
                    continue

                unity, _ = unit(p.data.view(p.size(0), -1))

                if stiefel and unity.size(0) <= unity.size(1):
                    rand_num = random.randint(1, 101)
                    if rand_num == 1:
                        unity = qr_retraction(unity)

                    G = p.grad.data.view(p.size(0), -1)          # raw Euclidean gradient
                    g = stiefel_proj_tan(unity, G)               # projected tangent / Riemannian gradient
                    lr = group['lr']

                    param_state = self.state[p]
                    if 'momentum_buffer' not in param_state:
                        param_state['momentum_buffer'] = torch.zeros_like(g.t())

                    V = param_state['momentum_buffer']
                    V = momentum * V - g.t()

                    MX = torch.mm(V, unity)
                    XMX = torch.mm(unity, MX)
                    XXMX = torch.mm(unity.t(), XMX)
                    W_hat = MX - 0.5 * XXMX
                    W = W_hat - W_hat.t()

                    t = 1.0 / (matrix_norm_one(W) + episilon)
                    alpha = min(t, lr)

                    p_new = Cayley_loop(unity.t(), W, V, alpha)
                    V_new = torch.mm(W, unity.t())  # n-by-p

                    p.data.copy_(p_new.view_as(p.data))
                    param_state['momentum_buffer'].copy_(V_new)

                else:
                    d_p = p.grad.data
                    if weight_decay != 0:
                        d_p = d_p.add(p.data, alpha=weight_decay)

                    if momentum != 0:
                        param_state = self.state[p]
                        if 'momentum_buffer' not in param_state:
                            buf = param_state['momentum_buffer'] = d_p.clone()
                        else:
                            buf = param_state['momentum_buffer']
                            buf.mul_(momentum).add_(d_p, alpha=(1 - dampening))

                        if nesterov:
                            d_p = d_p.add(buf, alpha=momentum)
                        else:
                            d_p = buf

                    p.data.add_(d_p, alpha=-group['lr'])

        return loss


class AdamG(Optimizer):
    r"""This optimizer updates variables with two different routines
        based on the boolean variable 'stiefel'.

        If stiefel is True, the variables will be updated by Adam-G.

        If stiefel is False, the variables will be updated by SGD-like momentum
        fallback as in the original code.

    Args:
        params (iterable): iterable of parameters to optimize or dicts defining
            parameter groups

        -- common parameters
        lr (float): learning rate
        momentum (float, optional): momentum factor (default: 0)
        stiefel (bool, optional): whether to use Adam-G (default: False)

        -- parameters in case stiefel is False
        weight_decay (float, optional): weight decay (L2 penalty) (default: 0)
        dampening (float, optional): dampening for momentum (default: 0)
        nesterov (bool, optional): enables Nesterov momentum (default: False)

        -- parameters in case stiefel is True
        beta2 (float, optional): exponential decay rate for second moment (default: 0.99)
        epsilon (float, optional): small constant for numerical stability (default: 1e-8)
        omega (float, optional): orthogonality regularization factor (default: 0)
        grad_clip (float, optional): threshold for gradient norm clipping (default: None)
    """

    def __init__(self, params, lr=required, momentum=0, dampening=0,
                 weight_decay=0, nesterov=False,
                 stiefel=False, beta2=0.99, epsilon=1e-8, omega=0, grad_clip=None):
        defaults = dict(
            lr=lr,
            momentum=momentum,
            dampening=dampening,
            weight_decay=weight_decay,
            nesterov=nesterov,
            stiefel=stiefel,
            beta2=beta2,
            epsilon=epsilon,
            omega=omega,
            grad_clip=grad_clip
        )
        if nesterov and (momentum <= 0 or dampening != 0):
            raise ValueError("Nesterov momentum requires a momentum and zero dampening")
        super(AdamG, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(AdamG, self).__setstate__(state)
        for group in self.param_groups:
            group.setdefault('nesterov', False)

    @torch.no_grad()
    def step(self, closure=None):
        """Performs a single optimization step."""
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            stiefel = group['stiefel']
            momentum = group['momentum']
            weight_decay = group['weight_decay']
            dampening = group['dampening']
            nesterov = group['nesterov']

            for p in group['params']:
                if p.grad is None:
                    continue

                beta1 = momentum
                beta2 = group['beta2']
                epsilon = group['epsilon']

                unity, _ = unit(p.data.view(p.size(0), -1))

                if stiefel and unity.size(0) <= unity.size(1):
                    rand_num = random.randint(1, 101)
                    if rand_num == 1:
                        unity = qr_retraction(unity)

                    G = p.grad.data.view(p.size(0), -1)          # raw Euclidean gradient
                    g = stiefel_proj_tan(unity, G)               # tangent-projected gradient

                    param_state = self.state[p]
                    if 'm_buffer' not in param_state:
                        size = p.size()
                        param_state['m_buffer'] = torch.zeros(
                            int(np.prod(size[1:])),
                            size[0],
                            device=p.device,
                            dtype=p.dtype
                        )
                        param_state['v_buffer'] = torch.zeros(
                            1, device=p.device, dtype=p.dtype
                        )
                        param_state['beta1_power'] = beta1
                        param_state['beta2_power'] = beta2

                    m = param_state['m_buffer']
                    v = param_state['v_buffer']
                    beta1_power = param_state['beta1_power']
                    beta2_power = param_state['beta2_power']

                    mnew = beta1 * m + (1.0 - beta1) * g.t()
                    vnew = beta2 * v + (1.0 - beta2) * (torch.norm(g) ** 2)

                    mnew_hat = mnew / (1 - beta1_power)
                    vnew_hat = vnew / (1 - beta2_power)

                    MX = torch.matmul(mnew_hat, unity)
                    XMX = torch.matmul(unity, MX)
                    XXMX = torch.matmul(unity.t(), XMX)
                    W_hat = MX - 0.5 * XXMX
                    W = (W_hat - W_hat.t()) / vnew_hat.add(epsilon).sqrt()

                    t = 1.0 / (matrix_norm_one(W) + episilon)
                    alpha = min(t, group['lr'])

                    p_new = Cayley_loop(unity.t(), W, mnew, -alpha)

                    p.data.copy_(p_new.view_as(p.data))
                    mnew = torch.matmul(W, unity.t()) * vnew_hat.add(epsilon).sqrt() * (1 - beta1_power)

                    m.copy_(mnew)
                    v.copy_(vnew)

                    param_state['beta1_power'] *= beta1
                    param_state['beta2_power'] *= beta2

                else:
                    d_p = p.grad.data
                    if weight_decay != 0:
                        d_p = d_p.add(p.data, alpha=weight_decay)

                    if momentum != 0:
                        param_state = self.state[p]
                        if 'momentum_buffer' not in param_state:
                            buf = param_state['momentum_buffer'] = d_p.clone()
                        else:
                            buf = param_state['momentum_buffer']
                            buf.mul_(momentum).add_(d_p, alpha=(1 - dampening))

                        if nesterov:
                            d_p = d_p.add(buf, alpha=momentum)
                        else:
                            d_p = buf

                    p.data.add_(d_p, alpha=-group['lr'])

        return loss


class MAdaGradG(Optimizer):
    r"""
    Riemannian AdaGrad-Norm on the Stiefel manifold using a Cayley transform sequence.

    For Stiefel parameters:
        beta_{k+1} = beta_k + ||grad f(X_k)||_F^2
        alpha_k    = lr / sqrt(beta_{k+1})
        X_{k+1}    = Cayley_retraction(X_k, -alpha_k * grad f(X_k))

    For non-Stiefel parameters:
        vanilla SGD step.
    """

    def __init__(self, params, lr=required, stiefel=False,
                 weight_decay=0.0, eps=1e-8, reorthogonalize=False):
        defaults = dict(
            lr=lr,
            stiefel=stiefel,
            weight_decay=weight_decay,
            eps=eps,
            reorthogonalize=reorthogonalize,
        )
        super(MAdaGradG, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(MAdaGradG, self).__setstate__(state)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            stiefel = group['stiefel']
            lr = group['lr']
            weight_decay = group['weight_decay']
            eps = group['eps']
            reorthogonalize = group['reorthogonalize']

            for p in group['params']:
                if p.grad is None:
                    continue

                g = p.grad.data

                # Euclidean fallback for ordinary parameters
                if not stiefel:
                    if weight_decay != 0:
                        g = g.add(p.data, alpha=weight_decay)
                    p.data.add_(g, alpha=-lr)
                    continue

                # Flatten weight to p-by-n, as in your current code
                Y, _ = unit(p.data.view(p.size(0), -1))
                if Y.size(0) > Y.size(1):
                    # same convention as your existing code:
                    # only use Stiefel branch when rows <= cols
                    if weight_decay != 0:
                        g = g.add(p.data, alpha=weight_decay)
                    p.data.add_(g, alpha=-lr)
                    continue

                G = g.view(p.size(0), -1)

                # Riemannian gradient on Stiefel (same p-by-n layout as Y)
                rgrad = stiefel_proj_tan(Y, G)

                # Optional regularization on ambient gradient before projection
                if weight_decay != 0:
                    rgrad = rgrad + weight_decay * Y

                grad_norm_sq = torch.sum(rgrad * rgrad)

                state = self.state[p]
                if 'beta_buffer' not in state:
                    state['beta_buffer'] = torch.zeros(
                        1, device=p.device, dtype=p.dtype
                    )

                beta = state['beta_buffer']
                beta.add_(grad_norm_sq)

                alpha = lr / torch.sqrt(beta + eps)

                # Convert tangent vector to n-by-p form for Cayley_loop
                V = -rgrad.t()  # n-by-p
                X = Y.t()       # n-by-p

                # Build skew-symmetric W exactly in your project style
                MX = torch.matmul(V, Y)          # n-by-n
                XMX = torch.matmul(Y, MX)        # p-by-n
                XXMX = torch.matmul(Y.t(), XMX)  # n-by-n
                W_hat = MX - 0.5 * XXMX
                W = W_hat - W_hat.t()            # n-by-n skew-symmetric

                # Stability safeguard, same spirit as SGDG/AdamG
                t = 1.0 / (matrix_norm_one(W) + eps)
                step_size = torch.minimum(alpha, t)

                # Cayley transform sequence update
                Y_new = Cayley_loop(X, W, V, step_size)

                # Optional exact cleanup if you want it
                if reorthogonalize:
                    Y_new = qr_retraction(Y_new)

                p.data.copy_(Y_new.view_as(p.data))

        return loss


In [ ]:
LR_STIEFEL = 0.2
LR_EUCLID = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

# learning-rate scheduler settings
LR_MILESTONES = [30, 60, 90]
LR_GAMMA = 0.2

stiefel_optimizer = SGDG(
    [
        {
            "params": stiefel_params,
            "lr": LR_STIEFEL,
            "stiefel": True,
            "weight_decay": WEIGHT_DECAY,

        }
    ]
)

euclid_optimizer = torch.optim.SGD(
    euclid_params,
    lr=LR_EUCLID,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

stiefel_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    stiefel_optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

euclid_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    euclid_optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

print("stiefel_optimizer =", stiefel_optimizer)
print("euclid_optimizer =", euclid_optimizer)

stiefel_optimizer = SGDG (
Parameter Group 0
    dampening: 0
    eps: 1e-08
    grad_clip: None
    lr: 0.005
    momentum: 0
    nesterov: False
    omega: 0
    reorthogonalize: True
    stiefel: True
    weight_decay: 0.0
)
euclid_optimizer = SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.0001
)


In [ ]:
# Define loss function
criterion = nn.CrossEntropyLoss() # Cross Entropy Loss

In [ ]:
# set the number of training epochs
num_epochs = 100

print("Starting training...")
print("Device:", device)
print("Number of epochs:", num_epochs)
print("Batch size:", batch_size)

In [ ]:
# create empty lists to store the average training loss and accuracy per epoch
train_losses = []
train_accuracies = []

# create empty lists to store the average validation loss and accuracy per epoch
val_losses = []
val_accuracies = []

# Move model to the correct device
model.to(device)

# ------------------------------------------------------------
# start the outer training loop over epochs
# ------------------------------------------------------------
for epoch in range(num_epochs):
    # put the model in training mode
    model.train()

    # initialize running training statistics
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    # --------------------------------------------------------
    # training phase
    # --------------------------------------------------------
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # clear old gradients from both optimizers
        stiefel_optimizer.zero_grad()
        euclid_optimizer.zero_grad()

        # forward pass
        outputs = model(images)

        # compute loss
        loss = criterion(outputs, labels)

        # backward pass
        loss.backward()

        # optimizer steps
        stiefel_optimizer.step()
        euclid_optimizer.step()

        # accumulate training statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, dim=1)
        running_correct += (predicted == labels).sum().item()
        running_total += labels.size(0)

    # average training metrics
    epoch_train_loss = running_loss / running_total
    epoch_train_acc = running_correct / running_total

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)

    # --------------------------------------------------------
    # validation phase
    # --------------------------------------------------------
    model.eval()

    val_running_loss = 0.0
    val_running_correct = 0
    val_running_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, dim=1)
            val_running_correct += (predicted == labels).sum().item()
            val_running_total += labels.size(0)

    # average validation metrics
    epoch_val_loss = val_running_loss / val_running_total
    epoch_val_acc = val_running_correct / val_running_total

    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)

    # update learning-rate schedulers after each epoch
    stiefel_scheduler.step()
    euclid_scheduler.step()

    # get current learning rates for printing
    current_lr_stiefel = stiefel_optimizer.param_groups[0]["lr"]
    current_lr_euclid = euclid_optimizer.param_groups[0]["lr"]


    print(
    f"Epoch [{epoch + 1}/{num_epochs}] | "
    f"LR_Euclid: {current_lr_euclid:.6f} | "
    f"LR_Stiefel: {current_lr_stiefel:.6f} | "
    f"Train Loss: {epoch_train_loss:.4f} | "
    f"Train Acc: {epoch_train_acc:.4f} | "
    f"Val Loss: {epoch_val_loss:.4f} | "
    f"Val Acc: {epoch_val_acc:.4f}"
)

Epoch [1/15] | Train Loss: 1.3346 | Train Acc: 0.5184 | Val Loss: 1.1204 | Val Acc: 0.6066
Epoch [2/15] | Train Loss: 0.8260 | Train Acc: 0.7093 | Val Loss: 0.7518 | Val Acc: 0.7328
Epoch [3/15] | Train Loss: 0.6263 | Train Acc: 0.7833 | Val Loss: 0.6334 | Val Acc: 0.7800
Epoch [4/15] | Train Loss: 0.5027 | Train Acc: 0.8234 | Val Loss: 0.5623 | Val Acc: 0.8118
Epoch [5/15] | Train Loss: 0.4159 | Train Acc: 0.8555 | Val Loss: 0.7443 | Val Acc: 0.7840
Epoch [6/15] | Train Loss: 0.3536 | Train Acc: 0.8764 | Val Loss: 0.4745 | Val Acc: 0.8450
Epoch [7/15] | Train Loss: 0.2862 | Train Acc: 0.8981 | Val Loss: 0.4430 | Val Acc: 0.8547
Epoch [8/15] | Train Loss: 0.2428 | Train Acc: 0.9133 | Val Loss: 0.4941 | Val Acc: 0.8458
Epoch [9/15] | Train Loss: 0.2007 | Train Acc: 0.9296 | Val Loss: 0.4558 | Val Acc: 0.8661


KeyboardInterrupt: 

In [ ]:
model.eval()

# initialize running sum of test loss
test_running_loss = 0.0

# initialize number of correct predictions on the test set
test_running_correct = 0

# initialize total number of test samples
test_running_total = 0

# CIFAR-10 class names for optional per-class reporting
classes = (
    'plane', 'car', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)

# create dictionaries to count correct predictions per class
class_correct = {cls_name: 0 for cls_name in classes}

# create dictionaries to count total samples per class
class_total = {cls_name: 0 for cls_name in classes}

# disable gradient tracking during testing
# this saves memory and speeds up evaluation
with torch.no_grad():
    # loop over test mini-batches
    for images, labels in test_loader:
        # move images to the selected device
        images = images.to(device)

        # move labels to the selected device
        labels = labels.to(device)

        # forward pass through the model
        outputs = model(images)

        # compute the batch loss
        loss = criterion(outputs, labels)

        # add batch loss weighted by batch size
        test_running_loss += loss.item() * images.size(0)

        # get predicted class indices from the output logits
        _, predicted = torch.max(outputs, dim=1)

        # count how many predictions are correct in this batch
        test_running_correct += (predicted == labels).sum().item()

        # update total number of tested samples
        test_running_total += labels.size(0)

        # update per-class counts
        for true_label, pred_label in zip(labels, predicted):
            # convert label index to Python integer
            true_idx = true_label.item()

            # map class index to class name
            cls_name = classes[true_idx]

            # count one more sample for this true class
            class_total[cls_name] += 1

            # if prediction is correct, count one more correct sample
            if pred_label.item() == true_idx:
                class_correct[cls_name] += 1

# compute average test loss
test_loss = test_running_loss / test_running_total

# compute average test accuracy
test_accuracy = test_running_correct / test_running_total

# print overall test metrics
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
results = pd.DataFrame({
    "epoch": list(range(1, len(train_losses) + 1)),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "val_loss": val_losses,
    "val_accuracy": val_accuracies,
})

results.to_csv("training_metrics.csv", index=False)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "stiefel_optimizer_state_dict": stiefel_optimizer.state_dict(),
        "euclid_optimizer_state_dict": euclid_optimizer.state_dict(),
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
    },
    "wrn_28_10_gs_fixed_checkpoint.pt",
)

print("Saved training_metrics.csv")
print("Saved wrn_28_10_gs_fixed_checkpoint.pt")

In [ ]:
#plotting
# ------------------------------------------------------------
# Plotting cell:
# show training and validation loss / accuracy
# ------------------------------------------------------------

import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

# ------------------------------------------------------------
# plot training and validation loss
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker='o', label='Train Loss')
plt.plot(epochs, val_losses, marker='s', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig("loss_curves.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# plot training and validation accuracy
# ------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_accuracies, marker='o', label='Train Accuracy')
plt.plot(epochs, val_accuracies, marker='s', label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)
plt.savefig("accuracy_curves.png", dpi=300, bbox_inches="tight")
plt.close()